<a href="https://colab.research.google.com/github/Adityadev1oper/GenAI-Projects/blob/main/Day1_classifyVsGenerate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn pandas matplotlib

In [2]:
import pandas as pd
data = {
    "report" : [
         "The front bumper has a deep crack",
        "The windshield is shattered",
        "The left door has a large dent",
        "The rear panel is scratched",
        "The hood is bent after the collision",
        "The side mirror is broken",
        "The car has no visible damage",
        "The vehicle is clean and undamaged",
        "The body panels are intact",
        "The car has no scratches or dents",
        "The vehicle looks normal and undamaged",
        "No visible cracks or broken parts"
    ],
    "label" :[
        "damaged", "damaged", "damaged", "damaged",
        "damaged", "damaged", "not_damaged", "not_damaged",
        "not_damaged", "not_damaged", "not_damaged",
        "not_damaged"
    ]
}

In [3]:
df = pd.DataFrame(data)

In [5]:
df.describe()

,report,label
count,12,12
unique,12,2
top,The front bumper has a deep crack,damaged
freq,1,6


In [6]:
df.head()

,report,label
0,The front bumper has a deep crack,damaged
1,The windshield is shattered,damaged
2,The left door has a large dent,damaged
3,The rear panel is scratched,damaged
4,The hood is bent after the collision,damaged


In [7]:
df.value_counts()

,,count
report,label,
No visible cracks or broken parts,not_damaged,1
The body panels are intact,not_damaged,1
The car has no scratches or dents,not_damaged,1
The car has no visible damage,not_damaged,1
The front bumper has a deep crack,damaged,1
The hood is bent after the collision,damaged,1
The left door has a large dent,damaged,1
The rear panel is scratched,damaged,1
The side mirror is broken,damaged,1


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase = True,
    stop_words = 'english',
    ngram_range= (1,2)
)

X = vectorizer.fit_transform(df['report'])
y = df['label']

print(X.shape[0],X.shape[1])


12 57


In [18]:
# Training the discriminative model = classify
from sklearn.linear_model import LogisticRegression
model =LogisticRegression()
model.fit(X,y)

LogisticRegression()

In [31]:
test_report = ["The tire is punctured",
                "The front bumper is cracked",
    "The vehicle has no visible damage",
    "The side door has a large dent"]
test_report_vextorized = vectorizer.transform(test_report)
prediction = model.predict(test_report_vextorized)
print(prediction)
probabilities = model.predict_proba(test_report_vextorized)
for report, prediction, probability in zip(
    test_report, prediction, probabilities
):
    print(f"\nReport: {report}")
    print(f"Prediction: {prediction}")

    class_probs = dict(zip(
        model.classes_,
        probability.round(3)
    ))
    print("Class probabilities:", class_probs)

['damaged' 'damaged' 'not_damaged' 'damaged']

Report: The tire is punctured
Prediction: damaged
Class probabilities: {'damaged': np.float64(0.509), 'not_damaged': np.float64(0.491)}

Report: The front bumper is cracked
Prediction: damaged
Class probabilities: {'damaged': np.float64(0.553), 'not_damaged': np.float64(0.447)}

Report: The vehicle has no visible damage
Prediction: not_damaged
Class probabilities: {'damaged': np.float64(0.395), 'not_damaged': np.float64(0.605)}

Report: The side door has a large dent
Prediction: damaged
Class probabilities: {'damaged': np.float64(0.591), 'not_damaged': np.float64(0.409)}


In [33]:
#Basic Generative model
import re
import random
from collections import defaultdict , Counter

In [34]:
corpus = [
    "The front bumper has a crack.",
    "The rear bumper has a dent.",
    "The windshield is broken.",
    "The left door has a scratch.",
    "The vehicle has visible damage.",
    "The vehicle has no visible damage.",
    "The car has a broken side mirror.",
    "The hood has a deep dent.",
    "The body panel has a scratch.",
    "The car looks clean and undamaged."
]

In [35]:
def tokenize(text):
   return re.findall(r"\b[\w']+\b|[.!?]", text.lower())

In [36]:
transitions = defaultdict(Counter)

In [37]:
for sentence in corpus:
    tokens = ["<START>"] + tokenize(sentence) + ["<END>"]

    for current_word, next_word in zip(tokens, tokens[1:]):
        transitions[current_word][next_word] += 1

print("Learned transitions:", len(transitions))
print("Possible continuations after 'the':")
print(transitions["the"])

Learned transitions: 31
Possible continuations after 'the':
Counter({'vehicle': 2, 'car': 2, 'front': 1, 'rear': 1, 'windshield': 1, 'left': 1, 'hood': 1, 'body': 1})


In [41]:
def generate_text(max_tokens=20):
    current = "<START>"
    output = []

    for _ in range(max_tokens):
        choices = transitions.get(current)

        if not choices:
            break

        words = list(choices.keys())
        counts = list(choices.values())

        next_word = random.choices(
            words,
            weights=counts,
            k=1
        )[0]

        if next_word == "<END>":
            break

        output.append(next_word)
        current = next_word

    text = " ".join(output)

    # Clean up spaces before punctuation.
    text = re.sub(r"\s+([.!?])", r"\1", text)

    return text.capitalize()


for i in range(5):
    print(f"{i + 1}. {generate_text()}")

1. The left door has no visible damage.
2. The vehicle has a scratch.
3. The car looks clean and undamaged.
4. The vehicle has a dent.
5. The body panel has a dent.
